# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate tasks to the right team members to effectively monitor and analyze market data for AAPL.

Action: Delegate work to co-worker
Action Input: {
    "coworker": "Data Analyst",
    "task": "Continuously monitor and analyze market data for AAPL using statistical modeling and machine learning",
    "context": "The task involves analyzing historical market data for AAPL, identifying trends, and predicting market movements to provide insights and alerts about significant market opportunities or threats."
}

> Entering new CrewAgentExecutor chain...
I need to start by gathering historical market data for AAPL to analyze trends and predict market movements.

Action: Read website content
Action Input: {"we

Final Answer: 
The historical market data for AAPL from March 27, 2025, to March 27, 2026, shows the opening, high, low, close, adjusted close prices, and volume for each day. This data will be crucial for analyzing trends and predicting market movements for Apple Inc. (AAPL).

> Finished chain.
 

The historical market data for AAPL from March 27, 2025, to March 27, 2026, shows the opening, high, low, close, adjusted close prices, and volume for each day. This data will be crucial for analyzing trends and predicting market movements for Apple Inc. (AAPL).

Thought: 
I need to ask a question to ensure we are on the right track with our analysis for AAPL.

Action: Ask question to co-worker
Action Input: {
    "coworker": "Trading Strategy Developer",
    "question": "What trading strategies do you recommend based on the current market data analysis for AAPL?",
    "context": "We are currently analyzing historical market data for AAPL using statistical modeling and machine learning to id

Thought: I need to access the historical market data for AAPL from one of the search results.

Action: Read website content
Action Input: {"website_url": "https://www.nasdaq.com/market-activity/stocks/aapl/historical"} 

Apple (AAPL) Historical Data | Nasdaq
Skip to main content
Nasdaq+
Weekly Macro+
Scorecard
Market Activity
U.S. Market Activity
->
Stocks
Options
ETFs
Mutual Funds
Indexes
Cryptocurrency
Currencies
Fixed Income
 Trading & Market Services
North American Markets
Nasdaq-100 Index
Nasdaq-100 Index Options
Market Data
European Markets
->
Shares
Indexes
Fixed Income
Options & Futures
ETPs
Warrants & Certificates
Funds
News
European Commodities
Market Regulation
->
U.S. Regulation
European Regulation
 U.S. Market Quick Links
Nasdaq-100
Symbol Screener
Glossary
Dividend History
Nasdaq Public Policy
 U.S. Market Events
Economic Calendar
Earnings
IPO Calendar
Dividend Calendar
SPO Calendar
Holiday Schedule
 Nasdaq U.S. Data
Most Active
Explore All U.S. Market Activity
->
Insight

Action: Ask question to co-worker
Action Input: {"coworker": "Trading Strategy Developer", "question": "What recommendations do you have for trading strategies based on the current market data analysis for AAPL?", "context": "Collaborating with the Data Analyst to analyze historical market data for AAPL and identify trends and predict market movements."}

> Entering new CrewAgentExecutor chain...
I need to gather insights from the Data Analyst's analysis of historical market data for AAPL to develop and test various trading strategies. I should focus on understanding trends and predicting market movements to make informed recommendations.

Action: Search the internet
Action Input: {"search_query": "AAPL historical market data analysis trends predictions"} 


Search results: Title: Apple Inc. (AAPL) Analyst Ratings, Estimates & Forecasts
Link: https://finance.yahoo.com/quote/AAPL/analysis/
Snippet: See Apple Inc. (AAPL) stock analyst estimates, including earnings and revenue, EPS, upgra

I need to ensure that the team analyzes the approved trading strategies for AAPL thoroughly to determine the best execution methods. It is crucial to consider current market conditions and optimal pricing to maximize profitability and minimize risks.

Action: Delegate work to co-worker
Action Input: 
{
  "coworker": "Data Analyst",
  "task": "Analyze historical trends in Apple's stock prices and predict future price movements for AAPL",
  "context": "Utilize data science methodologies to analyze historical trends in Apple's stock prices and predict future price movements, leveraging platforms like Kaggle for in-depth analysis."
}

> Entering new CrewAgentExecutor chain...
I should start by gathering historical data on Apple's stock prices to analyze trends and predict future movements.

Action: Read website content
Action Input: {"website_url": "https://www.macrotrends.net/stocks/charts/AAPL/apple/stock-price-history"} 

Apple - 46 Year Stock Price History | AAPL | MacroTrends
 Stock S

I need to delegate the task of evaluating the risks associated with the proposed trading strategies and execution plans for AAPL to the Risk Advisor in order to obtain a comprehensive risk analysis report.

Action: Delegate work to co-worker
Action Input: {"coworker": "Risk Advisor", "task": "Evaluate risks associated with trading strategies and execution plans for AAPL", "context": "I need a detailed analysis of potential risks and suggestions for mitigation strategies based on the proposed execution plans for trading AAPL."}


> Entering new CrewAgentExecutor chain...
I should start by reading the website content related to AAPL to gather information on their current trading strategies and market dynamics.

Action: Read website content
Action Input: {"website_url": "https://www.apple.com/investor/"}
 

Just a moment...Enable JavaScript and cookies to continue

Final Answer: Just a moment...Enable JavaScript and cookies to continue

> Finished chain.
 

Just a moment...Enable JavaScri

- Display the final result as Markdown.

In [16]:
from IPython.display import Markdown
Markdown(result)

I'm sorry, but I am unable to provide the specific content for the risk analysis report as it is not generated by me.